# The Purchasing Power Trap
Cyril James Lunzaga. Did Region XI statutory minimum-wage adjustments preserve purchasing power from January 2018 through December 2025? The legal wage benchmark is not observed earnings. Run the validated pipeline before opening this notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()), None)
if root is None or not (root / 'data' / 'processed' / 'monthly_wage_metrics.csv').exists():
    raise FileNotFoundError('Run the pipeline from the repository root before opening this notebook')
cpi = pd.read_csv(root / 'data' / 'processed' / 'monthly_cpi_metrics.csv', parse_dates=['month'])
wages = pd.read_csv(root / 'data' / 'processed' / 'monthly_wage_metrics.csv', parse_dates=['month'])
fies = pd.read_csv(root / 'reports' / 'tables' / 'fies_decile_composition_2023.csv')
episodes = pd.read_csv(root / 'reports' / 'tables' / 'below_reference_episodes.csv')
diagnostics = json.loads((root / 'reports' / 'tables' / 'validation_diagnostics.json').read_text())
temporal = json.loads((root / 'reports' / 'tables' / 'temporal_summary.json').read_text())

## Dataset coverage and validation
The analyzed core contains eight complete monthly CPI series and two purchasing-power series. An official FIES Table 7 aggregate provides 2023 composition across ten income deciles. The validation diagnostics document missingness, uniqueness, base-year, source checksum, wage-event, and published-share reconciliation tests.

In [ ]:
{name: diagnostics[name] for name in ('cpi', 'ppp', 'fies', 'base_year', 'ppp_reconciliation', 'wage_event_audit')}

## Statutory real-wage result
The main convention uses the wage in force at month end and deflates it by Davao's all-items CPI. The January 2018 comparison reference differs from the CPI's 2018 annual-average base.

In [ ]:
headline = wages.loc[wages['commodity'].eq('all_items')].set_index('month')
headline.loc[['2018-01-01', '2025-12-01'], ['nominal_wage_php', 'cpi', 'real_wage_2018_php', 'real_wage_change_percent']]

In [ ]:
image = plt.imread(root / 'reports' / 'figures' / 'real_wage_below_reference.png')
figure, axes = plt.subplots(figsize=(12, 6))
axes.imshow(image)
axes.axis('off')
plt.tight_layout()

## Temporal pattern and sensitivity
Below-reference months are grouped into consecutive episodes. The drawdown uses each month’s real wage relative to the highest earlier observed real wage. Wage-event windows are descriptive comparisons of adjacent months, not causal estimates.

In [ ]:
episodes, temporal['maximum_drawdown'], temporal['endpoint_decomposition']

In [ ]:
wages.loc[wages['month'].eq('2025-12-01') & wages['commodity'].isin(['all_items', 'food_and_beverages']), ['commodity', 'cpi', 'real_wage_2018_php', 'real_wage_change_percent']]

### Cross-sectional FIES evidence
Shares of *total family expenditure within an income decile* are descriptive. They are not the percentage of a national food category consumed by each decile.

In [ ]:
regional = fies.query("geography == 'davao_region' and income_decile in [1, 10]")
national = fies.query("geography == 'philippines' and income_decile in [1, 10]")
comparison = pd.concat([regional, national], ignore_index=True)
comparison[['geography', 'income_decile', 'food_and_beverages', 'selected_essential_share_percent']]

## FIES distributional context
The 2023 FIES values below are shares of aggregate family expenditure within each labeled per-capita-income decile. These families are not identified as minimum-wage earners. Survey shares are not monthly household spending or household-specific inflation.

In [ ]:
fies.loc[fies['geography'].eq('davao_region') & fies['income_decile'].isin([0, 1, 10]), ['income_decile', 'food_and_beverages', 'housing_and_utilities', 'transport', 'selected_essential_share_percent']]

In [ ]:
image = plt.imread(root / 'reports' / 'figures' / 'fies_food_share_davao_vs_philippines.png')
figure, axes = plt.subplots(figsize=(11, 5))
axes.imshow(image)
axes.axis('off')
plt.tight_layout()

## Conclusions and boundaries
The selected legal benchmark ends 11.19% above January 2018 in inflation-adjusted terms, so the full-window endpoint does not establish a sustained statutory purchasing-power trap. Seven months nevertheless fell below the initial real-wage level. The separate maximum drawdown from a *previous* monthly peak was 11.33%. FIES food expenditure shares were 58.3% for the lowest versus 23.9% for the highest Davao per-capita-income decile in 2023. None of these metrics measure actual wages, individual poverty or household-specific inflation, and no causal attribution to wage orders is attempted.